# 06 - Feature Engineering

## Objective

This notebook transforms the cleaned 2025 Airline On-Time Performance dataset into model-ready features for schedule-time flight-delay prediction.

Only information available before the scheduled departure of a flight will be used as predictive input. Variables generated during or after flight operations are excluded to prevent target leakage.

The feature-engineering process includes:

- Loading and validating the cleaned Delta table
- Creating the model-eligible flight population
- Engineering temporal features
- Engineering schedule features
- Selecting the final predictive variables
- Validating the engineered dataset
- Saving the feature dataset for model training


#### Load configuration and cleaned dataset

The feature-engineering process begins by loading the managed `flights_clean` Delta table produced by the data-cleaning notebook.

The source table is validated before transformations are applied. The raw and cleaned datasets remain unchanged throughout this notebook.


In [0]:
# Load the project configuration

from __future__ import annotations

from config import project_config as cfg
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T

print("Project configuration loaded successfully.")
print(f"Source table: {cfg.CLEAN_TABLE}")
print(f"Output table: {cfg.FEATURES_TABLE}")
print(f"Processed layer copy: {cfg.FEATURES_DELTA_PATH}")
print(f"Prediction target: {cfg.TARGET_COLUMN}")


In [0]:
# Load and validate the cleaned dataset

def require_table(table_name: str) -> None:
    """Raise an error when a required Unity Catalog table is unavailable."""
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run the data-cleaning notebook before continuing."
        )


require_table(cfg.CLEAN_TABLE)

df_clean: DataFrame = spark.table(cfg.CLEAN_TABLE)

clean_row_count = df_clean.count()
clean_column_count = len(df_clean.columns)

missing_columns = sorted(set(cfg.FEATURE_INPUT_COLUMNS) - set(df_clean.columns))

if missing_columns:
    raise ValueError(
        "Feature-engineering validation failed. "
        f"Missing required columns: {missing_columns}"
    )

print("Cleaned dataset loaded and validated successfully.")
print(f"Source table: {cfg.CLEAN_TABLE}")
print(f"Output table: {cfg.FEATURES_TABLE}")
print(f"Total records: {clean_row_count:,}")
print(f"Total columns: {clean_column_count}")
print(f"Prediction target: {cfg.TARGET_COLUMN}")


#### Create model-eligible dataset

The cleaned dataset contains all historical flight records, including cancelled and diverted flights.

For supervised machine learning, only flights that completed normal arrival operations are retained. Cancelled and diverted flights are excluded because the prediction target represents arrival delay for completed flights.


In [0]:
# Create model-eligible dataset

df_model = (
    df_clean
    .filter(
        (F.col(cfg.CANCELLED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
        & (F.col(cfg.DIVERTED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
    )
)

model_row_count = df_model.count()
removed_records = clean_row_count - model_row_count

model_summary = spark.createDataFrame(
    [
        (
            clean_row_count,
            model_row_count,
            removed_records,
        )
    ],
    schema="CLEAN_DATASET_ROWS long, MODEL_DATASET_ROWS long, EXCLUDED_RECORDS long",
)

display(model_summary)


#### Engineer temporal features

Temporal features capture calendar- and time-related patterns that may influence flight delays. These variables are derived exclusively from information available before the scheduled departure of a flight.

The following temporal features are created:

- `DEP_HOUR` — scheduled departure hour extracted from `CRS_DEP_TIME`
- `DEP_MINUTE` — scheduled departure minute extracted from `CRS_DEP_TIME`
- `IS_WEEKEND` — indicates whether the scheduled flight departs on a weekend
- `SEASON` — meteorological season derived from the flight month


In [0]:
# Engineer temporal features

df_features = df_model

df_features = df_features.withColumn(
    cfg.DEP_HOUR_COLUMN,
    F.floor(F.col(cfg.SCHEDULED_DEPARTURE_COLUMN) / 100).cast("int"),
)

df_features = df_features.withColumn(
    cfg.DEP_MINUTE_COLUMN,
    (F.col(cfg.SCHEDULED_DEPARTURE_COLUMN) % 100).cast("int"),
)

df_features = df_features.withColumn(
    cfg.IS_WEEKEND_COLUMN,
    F.when(
        F.col(cfg.DAY_OF_WEEK_COLUMN).isin(*cfg.WEEKEND_DAYS),
        F.lit(1),
    ).otherwise(F.lit(0)),
)

df_features = df_features.withColumn(
    cfg.SEASON_COLUMN,
    F.when(
        F.col(cfg.MONTH_COLUMN).isin(*cfg.SEASON_MONTH_GROUPS["Winter"]),
        "Winter",
    )
    .when(
        F.col(cfg.MONTH_COLUMN).isin(*cfg.SEASON_MONTH_GROUPS["Spring"]),
        "Spring",
    )
    .when(
        F.col(cfg.MONTH_COLUMN).isin(*cfg.SEASON_MONTH_GROUPS["Summer"]),
        "Summer",
    )
    .otherwise("Fall"),
)

print("Temporal features created successfully.")
print(f"Rows retained: {df_features.count():,}")
print(f"Columns after temporal engineering: {len(df_features.columns)}")


In [0]:
# Preview engineered temporal features

display(
    df_features.select(
        cfg.FLIGHT_DATE_COLUMN,
        cfg.MONTH_COLUMN,
        cfg.DAY_OF_WEEK_COLUMN,
        cfg.SCHEDULED_DEPARTURE_COLUMN,
        cfg.DEP_HOUR_COLUMN,
        cfg.DEP_MINUTE_COLUMN,
        cfg.IS_WEEKEND_COLUMN,
        cfg.SEASON_COLUMN,
    ).limit(20)
)


In [0]:
# Summarize temporal feature distributions

display(
    df_features
    .groupBy(cfg.SEASON_COLUMN)
    .count()
    .orderBy(cfg.SEASON_COLUMN)
)

display(
    df_features
    .groupBy(cfg.IS_WEEKEND_COLUMN)
    .count()
    .orderBy(cfg.IS_WEEKEND_COLUMN)
)

display(
    df_features
    .groupBy(cfg.DEP_HOUR_COLUMN)
    .count()
    .orderBy(cfg.DEP_HOUR_COLUMN)
)


#### Engineer schedule features

Schedule-related features describe operational characteristics that are known before departure.

The following schedule features are engineered:

- `TIME_OF_DAY` — categorizes scheduled departures into operational periods
- `FLIGHT_DISTANCE_CATEGORY` — groups flights into short-, medium-, and long-haul categories based on scheduled distance


In [0]:
# Engineer schedule features

df_features = df_features.withColumn(
    cfg.TIME_OF_DAY_COLUMN,
    F.when(F.col(cfg.DEP_HOUR_COLUMN).between(0, 5), "Overnight")
    .when(F.col(cfg.DEP_HOUR_COLUMN).between(6, 11), "Morning")
    .when(F.col(cfg.DEP_HOUR_COLUMN).between(12, 16), "Afternoon")
    .when(F.col(cfg.DEP_HOUR_COLUMN).between(17, 20), "Evening")
    .otherwise("Night"),
)

df_features = df_features.withColumn(
    cfg.FLIGHT_DISTANCE_CATEGORY_COLUMN,
    F.when(F.col(cfg.DISTANCE_COLUMN) < cfg.DISTANCE_SHORT_MAX_MILES, "Short")
    .when(
        F.col(cfg.DISTANCE_COLUMN) <= cfg.DISTANCE_MEDIUM_MAX_MILES,
        "Medium",
    )
    .otherwise("Long"),
)

print("Schedule features created successfully.")
print(f"Current columns: {len(df_features.columns)}")


In [0]:
# Summarize schedule feature distributions

display(
    df_features.groupBy(cfg.TIME_OF_DAY_COLUMN)
    .count()
    .orderBy(cfg.TIME_OF_DAY_COLUMN)
)

display(
    df_features.groupBy(cfg.FLIGHT_DISTANCE_CATEGORY_COLUMN)
    .count()
    .orderBy(cfg.FLIGHT_DISTANCE_CATEGORY_COLUMN)
)


#### Select final predictive variables

Following the project's target leakage restrictions, only variables available before the scheduled departure of a flight are retained for machine learning.

The final predictive dataset includes calendar, airline, route, geographic, schedule, engineered temporal, engineered schedule, and target variables.


In [0]:
# Select final predictive variables

df_ml = df_features.select(*cfg.MODEL_FEATURE_COLUMNS)

print("Final predictive dataset created successfully.")
print(f"Rows: {df_ml.count():,}")
print(f"Columns: {len(df_ml.columns)}")


In [0]:
# List final predictive variables

print("Final Predictive Variables")

for column in df_ml.columns:
    print(column)


#### Validate engineered dataset

The engineered dataset is validated before being used for machine learning model development.

The validation confirms total record count, feature count, missing values, and data types for the selected predictive variables.


In [0]:
# Summarize engineered dataset dimensions

validation_summary = spark.createDataFrame(
    [
        (
            df_ml.count(),
            len(df_ml.columns),
        )
    ],
    schema="TOTAL_RECORDS long, TOTAL_FEATURES long",
)

display(validation_summary)


In [0]:
# Summarize missing values in final features

null_summary = []
total_rows = df_ml.count()

for column_name in df_ml.columns:
    null_count = (
        df_ml
        .filter(F.col(column_name).isNull())
        .count()
    )

    null_percentage = round(
        (null_count / total_rows) * 100,
        4,
    )

    null_summary.append(
        (
            column_name,
            null_count,
            null_percentage,
        )
    )

null_summary_df = spark.createDataFrame(
    null_summary,
    [
        "COLUMN",
        "NULL_COUNT",
        "NULL_PERCENTAGE",
    ],
)

display(
    null_summary_df.orderBy(
        F.col("NULL_COUNT").desc()
    )
)


In [0]:
# Review final feature schema

schema_rows = []

for field in df_ml.schema.fields:
    schema_rows.append(
        (
            field.name,
            field.dataType.simpleString(),
            field.nullable,
        )
    )

schema_df = spark.createDataFrame(
    schema_rows,
    [
        "COLUMN_NAME",
        "DATA_TYPE",
        "NULLABLE",
    ],
)

display(schema_df)


#### Save feature dataset

The validated feature dataset is stored as a managed Delta table within Unity Catalog.

The resulting table will be used as the input dataset for model training.


In [0]:
# Register feature dataset as managed Delta table

(
    df_ml.writeTo(cfg.FEATURES_TABLE)
    .using("delta")
    .createOrReplace()
)

print("Feature dataset saved successfully.")
print(f"Table: {cfg.FEATURES_TABLE}")
print(f"Rows: {spark.table(cfg.FEATURES_TABLE).count():,}")


#### Store processed layer copy

In addition to registering the engineered dataset as a managed Unity Catalog table, a Delta copy is stored in the project's processed layer.


In [0]:
# Save feature dataset to processed layer

(
    df_ml.write
    .format("delta")
    .mode("overwrite")
    .save(cfg.FEATURES_DELTA_PATH)
)

print("Feature dataset copied to processed layer.")
print(f"Location: {cfg.FEATURES_DELTA_PATH}")


In [0]:
# Validate processed layer copy

df_feature_copy = spark.read.format("delta").load(cfg.FEATURES_DELTA_PATH)

print(f"Rows: {df_feature_copy.count():,}")
print(f"Columns: {len(df_feature_copy.columns)}")

display(df_feature_copy.limit(10))
